# 03 ファクターモデル

[研究案内](../資料/07_研究コードと成果物の案内.md) / [実装元](../src/quant_ai/factors.py)。VS Codeで関数名をF12、または下のセルで実装を表示。

## 1. 目的と仮説
複数のリスク要因と見かけのαを区別する。

## 2. 理論と数式
$r_i-r_f=\alpha+\beta_M f_M+\beta_S f_S+\beta_V f_V+\epsilon$。HAC付きOLS。

## 3. データの定義
seed7、600営業日、独立な3合成因子。実際のSMB/HML構築ではない。

## 4. 分析条件・パラメータ
真の負荷量1.1,0.3,-0.2、日次α0.0001、ノイズ標準偏差0.005。

## 5. 実装と実行
共通関数の実装も確認してから計算する。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/quant_ai").exists())
os.chdir(ROOT)
(ROOT / "分析結果/図").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("データ種別: 合成データ。実市場での収益性を示す結果ではありません。")


In [ ]:
import inspect
from IPython.display import Code
from quant_ai.factors import factor_regression
print(inspect.getsourcefile(factor_regression))
display(Code(inspect.getsource(factor_regression), language='python'))

In [ ]:
from quant_ai.factors import factor_regression
rng = np.random.default_rng(7)
factors = pd.DataFrame(rng.normal(0, 0.01, (600, 3)),
                       columns=["market", "size", "value"],
                       index=pd.bdate_range("2020-01-01", periods=600))
truth = pd.Series([1.1, 0.3, -0.2], index=factors.columns)
excess = 0.0001+factors@truth+rng.normal(0, 0.005, 600)
fit = factor_regression(excess, factors)
display(fit.summary())
display(pd.DataFrame({"truth": truth, "estimated": fit.params.drop("const")}))


## 6. 計算結果と可視化

In [ ]:
ax = fit.resid.plot(title="Factor residuals - synthetic")
ax.figure.savefig("分析結果/図/03_ファクターモデルの残差.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. 統計的解釈
偏回帰係数・残差と推定誤差を確認する。因子選択と共線性に注意する。

## 8. 金融工学・運用上の解釈
小さな残差でも因果関係は示さない。実際の因子は公表日と定義の確認が必要。

## 9. 限界と追加検証
## 解釈・限界・改善
残差が小さくても因果関係は示せません。実市場の公開因子を使う場合は配信日、通貨、頻度、
超過収益の定義、利用規約を確認してください。因子の事後選択はαの有意性を過大評価します。
改善案：ローリング推定、VIF、残差自己相関、頑健標準誤差、別期間での検証。